# Convert our Images into Text

In [1]:
import torch
from transformers import AutoProcessor, Blip2ForConditionalGeneration
import torch
import os
import requests
from PIL import Image
from io import BytesIO
from pathlib import Path
import pandas as pd
import numpy as np
from concurrent.futures import ThreadPoolExecutor, as_completed

In [2]:
print(f"CUDA available?: {torch.cuda.is_available()}")
print(f"Device Name: {torch.cuda.get_device_name()}")
device = "cuda:0"
torch.cuda.set_device(device)

CUDA available?: True
Device Name: AMD Radeon RX 9060 XT


<p style="color:red">Task for this notebook</p>

- [ ] comment code and explain (briefly) thought process

## Demo

In [3]:
processor = AutoProcessor.from_pretrained("Salesforce/blip2-opt-2.7b")
model = Blip2ForConditionalGeneration.from_pretrained(
    "Salesforce/blip2-opt-2.7b", 
    dtype=torch.float16
).to(device) 

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [4]:
img_url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/pipeline-cat-chonk.jpeg"
image = Image.open(requests.get(img_url, stream=True).raw).convert("RGB")

In [5]:
def generate_caption(image):
    inputs = processor(image, return_tensors="pt").to(device, torch.float16)

    generated_ids = model.generate(**inputs, max_new_tokens=30)
    generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
    
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    return generated_text

In [6]:
generate_caption(image)

/home/maharshii/miniconda3/envs/mlgeneral/lib/python3.10/site-packages/transformers/integrations/sdpa_attention.py:96: UserWarning: Flash Efficient attention on Current AMD GPU is still experimental. Enable it with TORCH_ROCM_AOTRITON_ENABLE_EXPERIMENTAL=1. (Triggered internally at /pytorch/aten/src/ATen/native/transformers/hip/sdp_utils.cpp:316.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(
/home/maharshii/miniconda3/envs/mlgeneral/lib/python3.10/site-packages/transformers/integrations/sdpa_attention.py:96: UserWarning: Mem Efficient attention on Current AMD GPU is still experimental. Enable it with TORCH_ROCM_AOTRITON_ENABLE_EXPERIMENTAL=1. (Triggered internally at /pytorch/aten/src/ATen/native/transformers/hip/sdp_utils.cpp:373.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


'a cat walking on snow in an enclosure'

## Convert Images into Text

### Load Datasets

In [7]:
data_dir = Path("../cleaned")  # change if your data directory is elsewhere

datasets = []

for file_name in data_dir.rglob("*.csv"):
    if file_name.stem.endswith("_images"):
        df = pd.read_csv(str(file_name))
        df._name = file_name.name[:-11]  # or file_name.stem, or str(file_name)
        datasets.append(df)

### Cleaning Datasets

In [8]:
for df in datasets:
    df["subreddit"] = df._name  # Assign subreddit name from file name to each dataframe
    df.drop(columns=["media_id", "image_type", "image_source"], inplace=True) # Drop media_id

KeyError: "['media_id', 'image_type', 'image_source'] not found in axis"

In [9]:
def parse_image(image_url, headers={"User-Agent": "Mozilla/5.0"}):
    resp = requests.get(image_url, headers=headers, timeout=5)
    resp.raise_for_status()
    image = Image.open(BytesIO(resp.content)).convert("RGBA")
    return image

In [10]:
def url_is_ok(url):
    try:
        parse_image(url)
        return True
    except Exception:
        return False

In [13]:
cpu_count = os.cpu_count() or 4
print("CPU cores:", cpu_count)

# Good starting points for IO-bound (network) work:
max_workers = cpu_count * 5   # or * 10 if your network / API can handle it
print("Suggested max_workers:", max_workers)

CPU cores: 12
Suggested max_workers: 60


In [14]:
cleaned_datasets = []
total_datasets = len(datasets)

for i, df in enumerate(datasets, start=1):
    original_len = len(df)
    bad_idx = []

    # run URL checks in parallel
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_idx = {
            executor.submit(url_is_ok, url): idx
            for idx, url in df["image_url"].items()
        }
        for future in as_completed(future_to_idx):
            idx = future_to_idx[future]
            ok = future.result()
            if not ok:
                bad_idx.append(idx)

    df = df.drop(index=bad_idx)
    cleaned_datasets.append(df)

    removed = len(bad_idx)
    pct_removed = (removed / original_len * 100) if original_len > 0 else 0.0

    print(f"Dataset {i}/{total_datasets}: removed {removed} images "
          f"({pct_removed:.2f}% removed).")
    print(f"Datasets left to filter through: {total_datasets - i}")

Dataset 1/78: removed 1 images (12.50% removed).
Datasets left to filter through: 77
Dataset 2/78: removed 6 images (54.55% removed).
Datasets left to filter through: 76
Dataset 3/78: removed 0 images (0.00% removed).
Datasets left to filter through: 75
Dataset 4/78: removed 14 images (2.97% removed).
Datasets left to filter through: 74
Dataset 5/78: removed 5 images (38.46% removed).
Datasets left to filter through: 73
Dataset 6/78: removed 48 images (6.71% removed).
Datasets left to filter through: 72
Dataset 7/78: removed 0 images (0.00% removed).
Datasets left to filter through: 71
Dataset 8/78: removed 0 images (0.00% removed).
Datasets left to filter through: 70
Dataset 9/78: removed 20 images (9.17% removed).
Datasets left to filter through: 69
Dataset 10/78: removed 3 images (15.79% removed).
Datasets left to filter through: 68
Dataset 11/78: removed 6 images (3.49% removed).
Datasets left to filter through: 67
Dataset 12/78: removed 4 images (40.00% removed).
Datasets left to 

/home/maharshii/miniconda3/envs/mlgeneral/lib/python3.10/site-packages/PIL/Image.py:3432: DecompressionBombWarning: Image size (114738774 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


Dataset 24/78: removed 29 images (5.82% removed).
Datasets left to filter through: 54
Dataset 25/78: removed 63 images (22.42% removed).
Datasets left to filter through: 53
Dataset 26/78: removed 1 images (100.00% removed).
Datasets left to filter through: 52
Dataset 27/78: removed 10 images (3.40% removed).
Datasets left to filter through: 51
Dataset 28/78: removed 1 images (14.29% removed).
Datasets left to filter through: 50
Dataset 29/78: removed 12 images (23.08% removed).
Datasets left to filter through: 49
Dataset 30/78: removed 43 images (9.01% removed).
Datasets left to filter through: 48


/home/maharshii/miniconda3/envs/mlgeneral/lib/python3.10/site-packages/PIL/Image.py:3432: DecompressionBombWarning: Image size (154716504 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


Dataset 31/78: removed 10 images (1.81% removed).
Datasets left to filter through: 47
Dataset 32/78: removed 152 images (26.81% removed).
Datasets left to filter through: 46
Dataset 33/78: removed 5 images (35.71% removed).
Datasets left to filter through: 45


...

Dataset 58/78: removed 18 images (4.00% removed).
Datasets left to filter through: 20
Dataset 59/78: removed 4 images (57.14% removed).
Datasets left to filter through: 19
Dataset 60/78: removed 42 images (13.38% removed).
Datasets left to filter through: 18
Dataset 61/78: removed 1 images (2.33% removed).
Datasets left to filter through: 17
Dataset 62/78: removed 2 images (1.54% removed).
Datasets left to filter through: 16
Dataset 63/78: removed 5 images (4.55% removed).
Datasets left to filter through: 15
Dataset 64/78: removed 4 images (4.21% removed).
Datasets left to filter through: 14
Dataset 65/78: removed 0 images (0.00% removed).
Datasets left to filter through: 13
Dataset 66/78: removed 13 images (2.50% removed).
Datasets left to filter through: 12
Dataset 67/78: removed 98 images (15.93% removed).
Datasets left to filter through: 11
Dataset 68/78: removed 21 images (1.64% removed).
Datasets left to filter through: 10


/home/maharshii/miniconda3/envs/mlgeneral/lib/python3.10/site-packages/PIL/Image.py:3432: DecompressionBombWarning: Image size (108000000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


Dataset 69/78: removed 2 images (1.44% removed).
Datasets left to filter through: 9
Dataset 70/78: removed 41 images (2.45% removed).
Datasets left to filter through: 8


/home/maharshii/miniconda3/envs/mlgeneral/lib/python3.10/site-packages/PIL/Image.py:3432: DecompressionBombWarning: Image size (96000000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
/home/maharshii/miniconda3/envs/mlgeneral/lib/python3.10/site-packages/PIL/Image.py:3432: DecompressionBombWarning: Image size (161678160 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


Dataset 71/78: removed 34 images (6.55% removed).
Datasets left to filter through: 7
Dataset 72/78: removed 2 images (0.78% removed).
Datasets left to filter through: 6
Dataset 73/78: removed 1 images (14.29% removed).
Datasets left to filter through: 5
Dataset 74/78: removed 0 images (0.00% removed).
Datasets left to filter through: 4
Dataset 75/78: removed 3 images (3.33% removed).
Datasets left to filter through: 3
Dataset 76/78: removed 5 images (22.73% removed).
Datasets left to filter through: 2
Dataset 77/78: removed 77 images (13.73% removed).
Datasets left to filter through: 1
Dataset 78/78: removed 9 images (1.88% removed).
Datasets left to filter through: 0


### Images -> Text Coversion

In [18]:
cpu_count = os.cpu_count() or 12
max_workers_fetch = cpu_count * 4      # ~48 workers on your CPU
max_workers_caption = 1                # GPU-bound, keep serial

captioned_datasets = []
total_datasets = len(cleaned_datasets)

for i, df in enumerate(cleaned_datasets, start=1):
    captions = {}

    # parallelize: fetch image + caption, small worker count (GPU-bound)
    def fetch_and_caption(idx, url):
        img = parse_image(url)          # network + PIL
        cap = generate_caption(img)     # GPU caption
        return idx, cap

    with ThreadPoolExecutor(max_workers=max_workers_caption) as executor:
        futures = [
            executor.submit(fetch_and_caption, idx, url)
            for idx, url in df["image_url"].items()
        ]

        for future in as_completed(futures):
            try:
                idx, cap = future.result()
            except Exception:
                # if captioning fails, store None (or "" if you prefer)
                idx, cap = None, None
            if idx is not None:
                captions[idx] = cap

    df["caption"] = df.index.map(captions.get)
    captioned_datasets.append(df)

    print(f"[CAPTION] Dataset {i}/{total_datasets} captioned "
          f"({total_datasets - i} datasets left).")

[CAPTION] Dataset 1/78 captioned (77 datasets left).
[CAPTION] Dataset 2/78 captioned (76 datasets left).
[CAPTION] Dataset 3/78 captioned (75 datasets left).
[CAPTION] Dataset 4/78 captioned (74 datasets left).
[CAPTION] Dataset 5/78 captioned (73 datasets left).
[CAPTION] Dataset 6/78 captioned (72 datasets left).
[CAPTION] Dataset 7/78 captioned (71 datasets left).
[CAPTION] Dataset 8/78 captioned (70 datasets left).
[CAPTION] Dataset 9/78 captioned (69 datasets left).
[CAPTION] Dataset 10/78 captioned (68 datasets left).
[CAPTION] Dataset 11/78 captioned (67 datasets left).
[CAPTION] Dataset 12/78 captioned (66 datasets left).
[CAPTION] Dataset 13/78 captioned (65 datasets left).
[CAPTION] Dataset 14/78 captioned (64 datasets left).
[CAPTION] Dataset 15/78 captioned (63 datasets left).
[CAPTION] Dataset 16/78 captioned (62 datasets left).
[CAPTION] Dataset 17/78 captioned (61 datasets left).
[CAPTION] Dataset 18/78 captioned (60 datasets left).
[CAPTION] Dataset 19/78 captioned (59

/home/maharshii/miniconda3/envs/mlgeneral/lib/python3.10/site-packages/PIL/Image.py:3432: DecompressionBombWarning: Image size (114738774 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


[CAPTION] Dataset 24/78 captioned (54 datasets left).
[CAPTION] Dataset 25/78 captioned (53 datasets left).
[CAPTION] Dataset 26/78 captioned (52 datasets left).
[CAPTION] Dataset 27/78 captioned (51 datasets left).
[CAPTION] Dataset 28/78 captioned (50 datasets left).
[CAPTION] Dataset 29/78 captioned (49 datasets left).
[CAPTION] Dataset 30/78 captioned (48 datasets left).


/home/maharshii/miniconda3/envs/mlgeneral/lib/python3.10/site-packages/PIL/Image.py:3432: DecompressionBombWarning: Image size (154716504 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


[CAPTION] Dataset 31/78 captioned (47 datasets left).
[CAPTION] Dataset 32/78 captioned (46 datasets left).
[CAPTION] Dataset 33/78 captioned (45 datasets left).


...

[CAPTION] Dataset 58/78 captioned (20 datasets left).
[CAPTION] Dataset 59/78 captioned (19 datasets left).
[CAPTION] Dataset 60/78 captioned (18 datasets left).
[CAPTION] Dataset 61/78 captioned (17 datasets left).
[CAPTION] Dataset 62/78 captioned (16 datasets left).
[CAPTION] Dataset 63/78 captioned (15 datasets left).
[CAPTION] Dataset 64/78 captioned (14 datasets left).
[CAPTION] Dataset 65/78 captioned (13 datasets left).
[CAPTION] Dataset 66/78 captioned (12 datasets left).
[CAPTION] Dataset 67/78 captioned (11 datasets left).
[CAPTION] Dataset 68/78 captioned (10 datasets left).


/home/maharshii/miniconda3/envs/mlgeneral/lib/python3.10/site-packages/PIL/Image.py:3432: DecompressionBombWarning: Image size (108000000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


[CAPTION] Dataset 69/78 captioned (9 datasets left).
[CAPTION] Dataset 70/78 captioned (8 datasets left).


/home/maharshii/miniconda3/envs/mlgeneral/lib/python3.10/site-packages/PIL/Image.py:3432: DecompressionBombWarning: Image size (161678160 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
/home/maharshii/miniconda3/envs/mlgeneral/lib/python3.10/site-packages/PIL/Image.py:3432: DecompressionBombWarning: Image size (96000000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


[CAPTION] Dataset 71/78 captioned (7 datasets left).
[CAPTION] Dataset 72/78 captioned (6 datasets left).
[CAPTION] Dataset 73/78 captioned (5 datasets left).
[CAPTION] Dataset 74/78 captioned (4 datasets left).
[CAPTION] Dataset 75/78 captioned (3 datasets left).
[CAPTION] Dataset 76/78 captioned (2 datasets left).
[CAPTION] Dataset 77/78 captioned (1 datasets left).
[CAPTION] Dataset 78/78 captioned (0 datasets left).


### Save Data Locally

In [34]:
data_dir = Path("../data")

csv_files = [
    str(file_name)[:8] + "cleaned/" + str(file_name)[8:] for file_name in data_dir.rglob("*.csv")
    if file_name.stem.endswith("_images")
]

In [ ]:
for df in datasets:
    df.to_csv(f"../cleaned/{df._name}_images.csv", index=False)
    print(f"Saved: ../cleaned/{df._name}_images.csv")

Saved: ../data/cleaned/sports_images.csv
Saved: ../data/cleaned/maybemaybemaybe_images.csv
Saved: ../data/cleaned/EatCheapAndHealthy_images.csv
Saved: ../data/cleaned/HistoryMemes_images.csv
Saved: ../data/cleaned/videos_images.csv
Saved: ../data/cleaned/pics_images.csv
Saved: ../data/cleaned/books_images.csv
Saved: ../data/cleaned/Jokes_images.csv
Saved: ../data/cleaned/Bitcoin_images.csv
Saved: ../data/cleaned/Showerthoughts_images.csv
Saved: ../data/cleaned/oddlysatisfying_images.csv
Saved: ../data/cleaned/history_images.csv
Saved: ../data/cleaned/cars_images.csv
Saved: ../data/cleaned/PremierLeague_images.csv
Saved: ../data/cleaned/StockMarket_images.csv
Saved: ../data/cleaned/Daytrading_images.csv
Saved: ../data/cleaned/MadeMeSmile_images.csv
Saved: ../data/cleaned/mildlyinfuriating_images.csv
Saved: ../data/cleaned/ThriftStoreHauls_images.csv
Saved: ../data/cleaned/dadjokes_images.csv
Saved: ../data/cleaned/foodhacks_images.csv
Saved: ../data/cleaned/unitedkingdom_images.csv
Save

### Merge Content into Original

In [8]:
for df in datasets:
    df["caption"] = "\nImage Description: " + df["caption"]

In [9]:
original_dir = Path("../data")  # change if your data directory is elsewhere
images_dir = datasets  # Path("../cleaned")

datasets_posts = {}
datasets_comments = {}

for file_name in original_dir.rglob("*.csv"):
    if file_name.stem.endswith("_posts"):
        df = pd.read_csv(str(file_name))
        datasets_posts[file_name.stem[:-6]] = df
    elif file_name.stem.endswith("_comments"):
        df = pd.read_csv(str(file_name))
        datasets_comments[file_name.stem[:-9]] = df

In [10]:
images_dir[0]

,subreddit,post_id,comment_id,image_index,image_url,caption
0,sports,1nn83c1,nfs6rbs,0,https://i.imgur.com/fEiAnK9.png,\nImage Description: a table with the names of...
1,sports,1nhmlbz,ned802x,0,https://i.redd.it/zzvcc5ox6qsy.jpg,\nImage Description: the office is a great show
2,sports,1n4dwk2,nblb7ty,0,https://i.imgur.com/CKp1Rup.png)...,\nImage Description: a computer screen showing...
3,sports,1n4dwk2,nblv56f,0,https://i.imgur.com/KriiYZS.png).,\nImage Description: a video game screen showi...
4,sports,1mymugi,naj7o7o,0,https://i.imgur.com/UpPrvIp.jpeg),\nImage Description: a black and white photo o...
5,sports,1mn7qem,n865828,0,https://i.imgur.com/S0eSAh6.gif),\nImage Description: a man with a caption that...
6,sports,1mjdw6b,n7b37iz,0,https://i.imgur.com/HJ1lIFs.png,\nImage Description: a man laying in bed with ...


In [11]:
datasets_posts["funny"]

,subreddit,post_id,post_title,post_score,post_url,post_content_url,post_text,timestamp,post_upvote_ratio,post_ups,post_total_awards_received,post_link_flair_text,post_author,post_num_comments,has_images,num_images,is_gallery,content_type
0,funny,160kuuf,] REMINDER] ANY political content will earn an...,697,https://reddit.com/r/funny/comments/160kuuf/re...,https://www.reddit.com/r/funny/comments/160kuu...,NaN,1.692927e+09,0.95,697,0,[Meta],funny_mod,1,False,0,False,text
1,funny,1npe17w,How did she miss?🤣,3423,https://reddit.com/r/funny/comments/1npe17w/ho...,https://v.redd.it/pm8xc67jk4rf1,NaN,1.758725e+09,0.94,3423,0,NaN,Master-Lunch3644,49,False,0,False,text
2,funny,1npcopx,One of the oddest movie trilogies of recent ti...,1727,https://reddit.com/r/funny/comments/1npcopx/on...,https://i.redd.it/l3a36lgza4rf1.jpeg,NaN,1.758722e+09,0.95,1727,0,NaN,IceBone,75,True,1,False,image
3,funny,1npgsd3,What are these exercises called?,1024,https://reddit.com/r/funny/comments/1npgsd3/wh...,https://v.redd.it/j7ftr5yx25rf1,NaN,1.758731e+09,0.93,1024,0,NaN,CakeFit7690,421,False,0,False,text
4,funny,1np8qxl,EA I WANT A REFUND NOW 🤣,2530,https://reddit.com/r/funny/comments/1np8qxl/ea...,https://v.redd.it/54sc4utzc3rf1,NaN,1.758710e+09,0.92,2530,0,NaN,Current-Assignment7,91,False,0,False,text
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,funny,1najlhy,kidney dance,266,https://reddit.com/r/funny/comments/1najlhy/ki...,https://v.redd.it/wlpltill3onf1,NaN,1.757218e+09,0.85,266,0,NaN,saifmahmud_,22,False,0,False,text
496,funny,1na71ys,Magnificent Toilet Experience....,1285,https://reddit.com/r/funny/comments/1na71ys/ma...,https://v.redd.it/4hieznqw8lnf1,NaN,1.757184e+09,0.76,1285,0,NaN,ZenKenShin,4,False,0,False,text
497,funny,1nauzx5,They set out the special donuts,31,https://reddit.com/r/funny/comments/1nauzx5/th...,https://i.redd.it/gmocbbhw8rnf1.jpeg,NaN,1.757256e+09,0.62,31,0,NaN,ColdH8WarmBlood,17,True,1,False,image
498,funny,1nbzz2z,Does she?,0,https://reddit.com/r/funny/comments/1nbzz2z/do...,https://i.redd.it/9a6f4mspa0of1.jpeg,NaN,1.757366e+09,0.44,0,0,NaN,TheRealBrandon2020,17,True,1,False,image


In [23]:
datasets_comments["funny"]

,post_id,comment_id,comment_text,comment_score,comment_author,comment_created_utc,parent_id,reply_to_id,comment_sentiment,has_images,num_images,image_urls,subreddit
0,160kuuf,jxmw8th,--- \n>✨⭐ **Don't miss [our 50-million-su...,1,AutoModerator,1.692927e+09,t3_160kuuf,160kuuf,NaN,False,0,NaN,funny
1,1npe17w,nfyf40y,--- \n\n>This is a friendly reminder to [...,1,AutoModerator,1.758725e+09,t3_1npe17w,1npe17w,NaN,False,0,NaN,funny
2,1npe17w,nfyg5gk,Malcolm would be impressed.,128,TheBleeter,1.758725e+09,t3_1npe17w,1npe17w,NaN,False,0,NaN,funny
3,1npe17w,nfytjnm,https://youtu.be/w199OAIBnA0,22,Choz,1.758729e+09,t1_nfyg5gk,nfyg5gk,NaN,False,0,NaN,funny
4,1npe17w,nfyiaoe,Here’s your damn strike!,29,Worldfriend,1.758726e+09,t1_nfyg5gk,nfyg5gk,NaN,False,0,NaN,funny
...,...,...,...,...,...,...,...,...,...,...,...,...,...
12260,1n9yobt,ncq406a,Not gonna lie lol 😆,9,Kvnq_Stormy,1.757163e+09,t1_ncq3n0v,ncq3n0v,NaN,False,0,NaN,funny
12261,1n9yobt,nd5l5df,I wish i could post that hype gif that everyon...,1,locofspades,1.757366e+09,t1_ncq3n0v,ncq3n0v,NaN,False,0,NaN,funny
12262,1n9yobt,nct9roy,"Other cat was like “oh shit, a ghost.”",1,The_Amazing_Emu,1.757200e+09,t1_ncq3n0v,ncq3n0v,NaN,False,0,NaN,funny
12263,1n9yobt,ncsg3bs,I am also amazed by this. Never seen this before.,0,SilentFX,1.757190e+09,t1_ncq3n0v,ncq3n0v,NaN,False,0,NaN,funny


In [12]:
for df in images_dir:
    if 'comment_id' not in df.columns:
        df['comment_id'] = None

In [13]:
# Merge captions from images_dir back into original posts and comments
for img_df in images_dir:
    subreddit_name = img_df._name
    print(f"Merging captions for subreddit: {subreddit_name}")
    
    # Get post captions (where post_id is not null and comment_id is null)
    post_images = img_df[img_df['post_id'].notna() & img_df['comment_id'].isna()].copy()
    
    # Get comment captions (where comment_id is not null)
    comment_images = img_df[img_df['comment_id'].notna()].copy()
    
    # Merge post captions into posts dataframe
    if subreddit_name in datasets_posts and len(post_images) > 0:
        # Group captions by post_id (in case multiple images per post)
        post_captions = post_images.groupby('post_id')['caption'].apply(
            lambda x: ' '.join(x.dropna().astype(str))
        ).reset_index()
        post_captions.columns = ['post_id', 'image_captions']
        
        # Merge into posts
        datasets_posts[subreddit_name] = datasets_posts[subreddit_name].merge(
            post_captions, on='post_id', how='left'
        )
        
        # Append captions to post_text where available
        mask = datasets_posts[subreddit_name]['image_captions'].notna()
        datasets_posts[subreddit_name].loc[mask, 'post_text'] = (
            datasets_posts[subreddit_name].loc[mask, 'post_text'].fillna('') + 
            datasets_posts[subreddit_name].loc[mask, 'image_captions']
        )
        
        # Drop the temporary column
        datasets_posts[subreddit_name].drop(columns=['image_captions'], inplace=True)
        
        print(f"✓ Merged {len(post_captions)} post captions into {subreddit_name}_posts")
    
    # Merge comment captions into comments dataframe
    if subreddit_name in datasets_comments and len(comment_images) > 0:
        # Group captions by comment_id (in case multiple images per comment)
        comment_captions = comment_images.groupby('comment_id')['caption'].apply(
            lambda x: ' '.join(x.dropna().astype(str))
        ).reset_index()
        comment_captions.columns = ['comment_id', 'image_captions']
        
        # Merge into comments
        datasets_comments[subreddit_name] = datasets_comments[subreddit_name].merge(
            comment_captions, on='comment_id', how='left'
        )
        
        # Append captions to comment_text where available
        mask = datasets_comments[subreddit_name]['image_captions'].notna()
        datasets_comments[subreddit_name].loc[mask, 'comment_text'] = (
            datasets_comments[subreddit_name].loc[mask, 'comment_text'].fillna('') + 
            datasets_comments[subreddit_name].loc[mask, 'image_captions']
        )
        
        # Drop the temporary column
        datasets_comments[subreddit_name].drop(columns=['image_captions'], inplace=True)
        
        print(f"✓ Merged {len(comment_captions)} comment captions into {subreddit_name}_comments")

print("\n✅ All captions merged successfully!")

Merging captions for subreddit: sports
✓ Merged 7 comment captions into sports_comments
Merging captions for subreddit: maybemaybemaybe
✓ Merged 4 comment captions into maybemaybemaybe_comments
Merging captions for subreddit: EatCheapAndHealthy
✓ Merged 1 post captions into EatCheapAndHealthy_posts
Merging captions for subreddit: HistoryMemes
✓ Merged 448 post captions into HistoryMemes_posts
✓ Merged 9 comment captions into HistoryMemes_comments
Merging captions for subreddit: videos
✓ Merged 8 comment captions into videos_comments
Merging captions for subreddit: pics
✓ Merged 450 post captions into pics_posts
✓ Merged 7 comment captions into pics_comments
Merging captions for subreddit: books
✓ Merged 1 post captions into books_posts
Merging captions for subreddit: Jokes
✓ Merged 1 comment captions into Jokes_comments
Merging captions for subreddit: Bitcoin
✓ Merged 184 post captions into Bitcoin_posts
Merging captions for subreddit: Showerthoughts
✓ Merged 16 comment captions into S

/tmp/ipykernel_54516/2090079016.py:27: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['\nImage Description: a collage of pictures of a woman on a motorcycle, a dog and a truck'
 '\nImage Description: a man in uniform giving a speech at a podium'
 '\nImage Description: president trump and his staff pose for a photo in the oval office \nImage Description: president donald trump shakes hands with students at the white house \nImage Description: president donald trump with students at the white house \nImage Description: a group of young men holding an american flag'
 '\nImage Description: a black and white photo of three men and a woman'
 '\nImage Description: a man stands in the middle of a hole in the ground'
 '\nImage Description: a man is riding a bike in front of a building'
 '\nImage Description: a woman wearing a colorful scarf taking a selfie'
 '\nImage Description: an old black and white photo of a

✓ Merged 1 comment captions into dadjokes_comments
Merging captions for subreddit: foodhacks
✓ Merged 122 post captions into foodhacks_posts
✓ Merged 2 comment captions into foodhacks_comments
Merging captions for subreddit: unitedkingdom
✓ Merged 3 post captions into unitedkingdom_posts
✓ Merged 8 comment captions into unitedkingdom_comments
Merging captions for subreddit: RelationshipMemes
✓ Merged 480 post captions into RelationshipMemes_posts
Merging captions for subreddit: Art
✓ Merged 469 post captions into Art_posts
Merging captions for subreddit: IAmA
✓ Merged 162 post captions into IAmA_posts
✓ Merged 17 comment captions into IAmA_comments
Merging captions for subreddit: Entrepreneur
Merging captions for subreddit: space
✓ Merged 157 post captions into space_posts
✓ Merged 5 comment captions into space_comments
Merging captions for subreddit: worldnews
✓ Merged 6 comment captions into worldnews_comments
Merging captions for subreddit: indieheads
✓ Merged 23 post captions into 

/tmp/ipykernel_54516/2090079016.py:27: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['\nImage Description: cloning a game is perfectly legal the tevis company'
 '\nImage Description: statistics in eu young people at the age of 25 years old have a higher income than you do'
 "\nImage Description: when it's not your fault but you just happened to be at the wrong place and time"
 "\nImage Description: well it's the day of the rapture again | made w/ imgflip meme maker"
 '\nImage Description: when a youtube you loved starts making brain videos | image tagged in youtube,youtube,memes | made w/ imgflip meme maker'
 "\nImage Description: two pictures of tom and jerry with the caption saying, 'my mom thinks about who keeps using all the shampoo in the bathroom"
 "\nImage Description: a cartoon character is standing in front of a green sign that says, 'christians walking to work after being wrong'"
 '\nImage Des

✓ Merged 474 post captions into gifs_posts
✓ Merged 94 comment captions into gifs_comments
Merging captions for subreddit: Futurology
✓ Merged 1 comment captions into Futurology_comments
Merging captions for subreddit: strength_training
✓ Merged 1 comment captions into strength_training_comments
Merging captions for subreddit: funny
✓ Merged 281 post captions into funny_posts
✓ Merged 10 comment captions into funny_comments
Merging captions for subreddit: NetflixBestOf
✓ Merged 11 post captions into NetflixBestOf_posts
Merging captions for subreddit: 15minutefood
✓ Merged 374 post captions into 15minutefood_posts
✓ Merged 20 comment captions into 15minutefood_comments
Merging captions for subreddit: aww
✓ Merged 389 post captions into aww_posts
✓ Merged 112 comment captions into aww_comments
Merging captions for subreddit: Fitness
✓ Merged 13 comment captions into Fitness_comments
Merging captions for subreddit: Unexpected
✓ Merged 186 comment captions into Unexpected_comments
Merging 

/tmp/ipykernel_54516/2090079016.py:27: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['\nImage Description: a man is standing in front of a waterfall'
 '\nImage Description: lily pads in a pond with tall grasses'
 '\nImage Description: a black and white photo of a mountain and ocean'
 '\nImage Description: a dirt road in the middle of a field with a cloudy sky'
 '\nImage Description: an elephant with a leaf in its mouth'
 '\nImage Description: a black and white photo of a window with a view of a city'
 '\nImage Description: a river with a misty sky and a colorful sunset'
 '\nImage Description: a lake with mountains and clouds reflected in it'
 '\nImage Description: a large cathedral with a statue of a man'
 '\nImage Description: black and white photograph of trees in the forest'
 '\nImage Description: a person standing in front of a large organ'
 '\nImage Description: a white cat with yellow eyes sitting

✓ Merged 482 post captions into itookapicture_posts
Merging captions for subreddit: HumansBeingBros
✓ Merged 32 post captions into HumansBeingBros_posts
✓ Merged 11 comment captions into HumansBeingBros_comments
Merging captions for subreddit: WatchPeopleDieInside
✓ Merged 5 post captions into WatchPeopleDieInside_posts
✓ Merged 50 comment captions into WatchPeopleDieInside_comments
Merging captions for subreddit: movies
✓ Merged 38 post captions into movies_posts
✓ Merged 5 comment captions into movies_comments
Merging captions for subreddit: AskReddit
✓ Merged 5 comment captions into AskReddit_comments
Merging captions for subreddit: MakeupAddiction
✓ Merged 227 post captions into MakeupAddiction_posts
✓ Merged 81 comment captions into MakeupAddiction_comments
Merging captions for subreddit: math
✓ Merged 51 post captions into math_posts
✓ Merged 5 comment captions into math_comments
Merging captions for subreddit: AnimalsBeingBros
✓ Merged 86 post captions into AnimalsBeingBros_post

/tmp/ipykernel_54516/2090079016.py:27: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['\nImage Description: a mountain range with snow capped peaks in the distance'
 '\nImage Description: a rainbow is seen in front of a waterfall'
 '\nImage Description: a view of a lake and mountains with fog and red leaves'
 '\nImage Description: black and white photograph of trees in the forest'
 '\nImage Description: a lake with rocks and grass at sunset'
 '\nImage Description: the desert is covered with white rock formations'
 '\nImage Description: a view of the mountains and trees in the valley'
 '\nImage Description: a canyon with a road and trees in the foreground'
 '\nImage Description: a tree is in the water'
 '\nImage Description: moraine lake, banff national park, canada'
 '\nImage Description: a person standing on a sand dune with a milky way in the background'
 '\nImage Description: a mountain range with tre

In [15]:
# Save updated posts
for subreddit_name, df in datasets_posts.items():
    output_path = f"../cleaned/{subreddit_name}_posts.csv"
    df.to_csv(output_path, index=False)
    print(f"Saved: {output_path}")

# Save updated comments
for subreddit_name, df in datasets_comments.items():
    output_path = f"../cleaned/{subreddit_name}_comments.csv"
    df.to_csv(output_path, index=False)
    print(f"Saved: {output_path}")

Saved: ../cleaned/Art_posts.csv
Saved: ../cleaned/ChatGPT_posts.csv
Saved: ../cleaned/math_posts.csv
Saved: ../cleaned/ThriftStoreHauls_posts.csv
Saved: ../cleaned/memes_posts.csv
Saved: ../cleaned/HistoryMemes_posts.csv
Saved: ../cleaned/unitedkingdom_posts.csv
Saved: ../cleaned/pics_posts.csv
Saved: ../cleaned/AnimalsBeingBros_posts.csv
Saved: ../cleaned/BikiniBottomTwitter_posts.csv
Saved: ../cleaned/RelationshipMemes_posts.csv
Saved: ../cleaned/NetflixBestOf_posts.csv
Saved: ../cleaned/Daytrading_posts.csv
Saved: ../cleaned/EatCheapAndHealthy_posts.csv
Saved: ../cleaned/MakeupAddiction_posts.csv
Saved: ../cleaned/mildlyinfuriating_posts.csv
Saved: ../cleaned/Survival_posts.csv
Saved: ../cleaned/strength_training_posts.csv
Saved: ../cleaned/WatchPeopleDieInside_posts.csv
Saved: ../cleaned/Bitcoin_posts.csv
Saved: ../cleaned/pettyrevenge_posts.csv
Saved: ../cleaned/news_posts.csv
Saved: ../cleaned/indieheads_posts.csv
Saved: ../cleaned/maybemaybemaybe_posts.csv
Saved: ../cleaned/Expe

Saved: ../cleaned/IAmA_posts.csv
Saved: ../cleaned/Entrepreneur_posts.csv
Saved: ../cleaned/tifu_posts.csv
Saved: ../cleaned/movies_posts.csv
Saved: ../cleaned/SkincareAddiction_posts.csv
Saved: ../cleaned/Unexpected_posts.csv
Saved: ../cleaned/Showerthoughts_posts.csv
Saved: ../cleaned/science_posts.csv
Saved: ../cleaned/DIY_posts.csv
Saved: ../cleaned/foodhacks_posts.csv
Saved: ../cleaned/GetMotivated_posts.csv
Saved: ../cleaned/backpacking_posts.csv
Saved: ../cleaned/todayilearned_posts.csv
Saved: ../cleaned/europe_posts.csv
Saved: ../cleaned/PremierLeague_posts.csv
Saved: ../cleaned/oddlysatisfying_posts.csv
Saved: ../cleaned/anime_irl_posts.csv
Saved: ../cleaned/dadjokes_posts.csv
Saved: ../cleaned/explainlikeimfive_posts.csv
Saved: ../cleaned/philosophy_posts.csv
Saved: ../cleaned/books_posts.csv
Saved: ../cleaned/EarthPorn_posts.csv
Saved: ../cleaned/psychology_posts.csv
Saved: ../cleaned/boardgames_posts.csv
Saved: ../cleaned/iphone_posts.csv
Saved: ../cleaned/IAmA_comments.csv